# Email Query & Reply Agent (Gmail + Gemma 3)

This notebook builds an agent that:
1. Takes a user query (e.g. sender email or subject keywords) to find a specific email in Gmail.
2. Reads and understands that email using a local Hugging Face LLM (Gemma 3 4B Instruct).
3. Drafts a reply.
4. Lets the user **send** the reply, **regenerate** it with feedback, **edit** it manually, or **quit**
   without sending — nothing is sent without explicit user confirmation.

**Requirements:**
- A Hugging Face account with access to the gated `google/gemma-3-4b-it` model, and an HF access token.
- A Google Cloud OAuth **client_secret.json** file for a project with the Gmail API enabled
  (scope: `gmail.modify`).
- A GPU runtime is strongly recommended for running the 4B-parameter model.


## 1. Install Dependencies

In [ ]:
# Libraries for running the local Hugging Face model
!pip install -q transformers accelerate bitsandbytes huggingface_hub datasets peft trl


## 2. Load the Language Model (Gemma 3 4B Instruct)

Model page: https://huggingface.co/google/gemma-3-4b-it

This model is **gated** — you must request access on the model page and authenticate with a
Hugging Face token (set `HF_TOKEN` as an environment variable/secret, or log in interactively below)
before it can be downloaded.


In [ ]:
from huggingface_hub import login

# Paste your Hugging Face access token below (or set the HF_TOKEN environment variable instead
# of hardcoding it here)
login(token="")


In [ ]:
# Load the Gemma 3 processor and model.
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto")

# Quick sanity check: ask the model a simple multimodal question using a sample image,
# to confirm the model and processor loaded correctly before moving on to the email task
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


## 3. Gmail Authentication

Installs the Google API client libraries and runs an OAuth flow so the agent can read and send
email on your behalf (scope: `gmail.modify`).


In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib


In [ ]:
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
import pickle

SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]

# Path to the OAuth client secrets JSON file downloaded from Google Cloud Console
# (APIs & Services > Credentials > OAuth 2.0 Client ID > Desktop app)
CLIENT_SECRETS_FILE = ""

flow = Flow.from_client_secrets_file(
    CLIENT_SECRETS_FILE,
    scopes=SCOPES,
    redirect_uri="urn:ietf:wg:oauth:2.0:oob"
)

# Manual "out-of-band" OAuth flow: open the URL, sign in, approve access, and paste
# back the authorization code shown by Google (used here because there's no local
# redirect server in a notebook environment)
auth_url, _ = flow.authorization_url(prompt="consent")
print("Go to this URL, sign in, and approve access:\n")
print(auth_url)

code = input("\nPaste the authorization code here: ")
flow.fetch_token(code=code)
creds = flow.credentials

# Cache credentials locally so future runs don't require re-authenticating
with open("token.pickle", "wb") as f:
    pickle.dump(creds, f)

gmail_service = build("gmail", "v1", credentials=creds)
print("\nGmail authenticated.")


## 4. Find the Target Email

Lets the user search Gmail with a query (e.g. `from:someone@example.com subject:pricing`), shows
the top matches, and asks the user to pick which one the agent should reply to.


In [ ]:
def find_email(query):
    """
    Searches Gmail using the given query string, displays up to 5 matching messages
    (sender, subject, date), and prompts the user to choose which one to act on.
    Returns the Gmail message ID of the chosen email, or None if no matches were found.
    """
    results = gmail_service.users().messages().list(userId="me", q=query, maxResults=5).execute()
    messages = results.get("messages", [])
    if not messages:
        print("No matching email found.")
        return None

    print(f"Found {len(messages)} matches:\n")
    for i, m in enumerate(messages):
        # Fetch only the metadata (Subject/From/Date) rather than the full message body,
        # since we just need enough info here to let the user identify the right email
        msg = gmail_service.users().messages().get(
            userId="me", id=m["id"], format="metadata",
            metadataHeaders=["Subject", "From", "Date"]
        ).execute()
        headers = {h["name"]: h["value"] for h in msg["payload"]["headers"]}
        print(f"[{i}] From: {headers.get('From')} | Subject: {headers.get('Subject')} | Date: {headers.get('Date')}")

    choice = int(input("\nWhich one? (enter number): "))
    return messages[choice]["id"]


search_query = input("Search for email (e.g. 'from:someone@example.com subject:pricing'): ")
email_id = find_email(search_query)
print(f"\nSelected message ID: {email_id}")


## 5. Fetch the Full Email Content

In [ ]:
import base64

def get_email_content(msg_id):
    """
    Fetches the full content of a Gmail message by ID and extracts the fields the
    agent needs: subject, sender, plain-text body, thread ID (for replying in-thread),
    and the message ID itself.
    """
    msg = gmail_service.users().messages().get(userId="me", id=msg_id, format="full").execute()
    headers = msg["payload"]["headers"]
    subject = next(h["value"] for h in headers if h["name"] == "Subject")
    sender = next(h["value"] for h in headers if h["name"] == "From")
    thread_id = msg["threadId"]

    def get_body(payload):
        """Recursively-aware helper: Gmail messages can be single-part (body directly
        on the payload) or multi-part (body nested inside 'parts'). This handles both,
        preferring the plain-text part when the message is multi-part."""
        if "parts" in payload:
            for part in payload["parts"]:
                if part["mimeType"] == "text/plain":
                    return base64.urlsafe_b64decode(part["body"]["data"]).decode("utf-8")
        elif "body" in payload and "data" in payload["body"]:
            return base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8")
        return ""

    body = get_body(msg["payload"])
    return {"subject": subject, "sender": sender, "body": body, "thread_id": thread_id, "msg_id": msg_id}


email_data = get_email_content(email_id)
print(email_data)


## 6. Generate a Reply with the LLM

Prompts Gemma 3 to draft a reply in a professional but warm tone. If `feedback` is provided
(from a previous round of user feedback), it's appended as revision instructions so the model
can regenerate an improved draft.


In [ ]:
EMAIL_SYSTEM_PROMPT = (
    "You are a professional assistant that writes clear, polite, and concise email replies. "
    "Match a professional but warm tone. Address the sender's message directly. "
    "Do not include a subject line, greeting placeholder, or signature — just the reply body."
)


def generate_reply(email_data, feedback=None):
    """
    Generates a draft reply to the given email using the loaded LLM.

    If `feedback` is provided, it's appended to the prompt as revision instructions
    (e.g. 'make it shorter', 'more formal'), allowing the agent to regenerate the
    draft based on the user's requested changes.
    """
    extra = f"\n\nRevision instructions: {feedback}" if feedback else ""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": EMAIL_SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": (
            f"From: {email_data['sender']}\n"
            f"Subject: {email_data['subject']}\n"
            f"Body: {email_data['body']}\n\n"
            f"Write a professional reply to this email.{extra}"
        )}]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=300)
    reply = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return reply.strip()


reply_text = generate_reply(email_data)
print(reply_text)


## 7. Send the Reply

In [ ]:
from email.mime.text import MIMEText

def send_reply(email_data, reply_text):
    """
    Sends the reply via the Gmail API, keeping it in the same thread as the original
    email (so it appears as a proper reply rather than a new, disconnected message).
    """
    message = MIMEText(reply_text)
    message["to"] = email_data["sender"]
    message["subject"] = "Re: " + email_data["subject"]
    message["In-Reply-To"] = email_data["msg_id"]
    message["References"] = email_data["msg_id"]

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    body = {"raw": raw, "threadId": email_data["thread_id"]}

    sent = gmail_service.users().messages().send(userId="me", body=body).execute()
    print(f"Sent. Message ID: {sent['id']}")


## 8. Human-in-the-Loop Confirmation Loop

The core control loop: the drafted reply is never sent automatically. The user must explicitly
choose to **(s)end**, **(r)egenerate** with feedback (the writer model drafts again), **(e)dit**
the reply manually, or **(q)uit** without sending anything.


In [ ]:
while True:
    print("\n--- Generated Reply ---")
    print(reply_text)
    print("------------------------")

    choice = input("\n(s)end, (r)egenerate with feedback, (e)dit manually, or (q)uit: ").strip().lower()

    if choice == "s":
        # User confirmed — send the reply and log it
        send_reply(email_data, reply_text)
        with open("sent_log.txt", "a") as f:
            f.write(f"To: {email_data['sender']}\nSubject: Re: {email_data['subject']}\n{reply_text}\n{'='*40}\n")
        break
    elif choice == "r":
        # User wants changes — collect feedback and regenerate the draft
        feedback = input("What should change? (e.g. 'shorter', 'more formal', 'mention the refund policy'): ")
        reply_text = generate_reply(email_data, feedback=feedback)
    elif choice == "e":
        # User wants to write the reply themselves instead of using the model's draft
        new_text = input("\nType your replacement reply (or press Enter to keep current): ")
        if new_text.strip():
            reply_text = new_text
    elif choice == "q":
        # User cancelled — nothing is sent, loop exits
        print("Cancelled. Nothing was sent.")
        break
    else:
        print("Invalid choice — type s, r, e, or q.")
